# 2.1 — Data Cleaning (Fixed)

**Perbaikan dari audit:**
- ❌ PCHIP smoothing dihapus — tidak membuat data sintetis antar titik tahunan
- ✅ Forward-carry (step function) untuk data tahunan — jujur apa adanya
- ✅ `x4_tpt_pct` diinterpolasi linear antara Feb & Agt (bukan ffill 5 bulan)
- ✅ Filter `tahun > 2021` (menghilangkan data COVID awal)

**Input:** `1_data_gathering/output/1_raw_panel_data.csv`

**Output:** `2_data_preprocessing/output/2.1_cleaned_data.csv`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / '1_data_gathering' / 'output' / '1_raw_panel_data.csv').exists():
            return p
    raise FileNotFoundError('Could not find 1_raw_panel_data.csv')

ROOT = find_project_root(Path.cwd())
input_path = ROOT / '1_data_gathering' / 'output' / '1_raw_panel_data.csv'
output_dir = ROOT / '2_data_preprocessing' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / '2.1_cleaned_data.csv'

df = pd.read_csv(input_path)
df['tanggal'] = pd.to_datetime(df['tanggal'])
df = df.sort_values(by=['provinsi_id', 'tanggal']).reset_index(drop=True)

print(f'Loaded: {input_path}')
print(f'Raw shape: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Date range: {df["tanggal"].min()} - {df["tanggal"].max()}')
print(f'Provinces: {df["provinsi_id"].nunique()}')

In [ ]:
# --- Missing value summary SEBELUM cleaning ---
print('=== Missing Values (Raw) ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
summary = pd.DataFrame({'missing': missing, 'pct': missing_pct})
display(summary[summary['missing'] > 0])

In [ ]:
# --- Filter periode: tahun > 2021 (menghilangkan COVID awal) ---
df_filtered = df[df['tahun'] > 2021].copy()
print(f'Filtered shape (tahun > 2021): {df_filtered.shape[0]:,} rows x {df_filtered.shape[1]} cols')

# --- FIX 1: x4_tpt_pct — interpolasi linear per provinsi ---
# Data TPT hanya tersedia di bulan 2 (Feb) dan 8 (Agt)
# Interpolasi linear antara titik-titik tersebut (bukan ffill konstan)
df_filtered['x4_tpt_pct'] = (
    df_filtered.groupby('provinsi_id')['x4_tpt_pct']
    .transform(lambda s: s.interpolate(method='linear', limit_direction='both'))
)
missing_x4_after = df_filtered['x4_tpt_pct'].isna().sum()
print(f'Missing x4_tpt_pct after linear interpolation: {missing_x4_after}')

# --- FIX 2: x5_penetrasi_internet — forward-fill per provinsi (data tahunan) ---
# Tidak ada data 2021, jadi bfill dari 2022 untuk awal series
df_filtered['x5_penetrasi_internet_pct'] = (
    df_filtered.groupby('provinsi_id')['x5_penetrasi_internet_pct']
    .transform(lambda s: s.ffill().bfill())
)
missing_x5_after = df_filtered['x5_penetrasi_internet_pct'].isna().sum()
print(f'Missing x5_penetrasi_internet_pct after fill: {missing_x5_after}')

In [ ]:
# --- FIX 3: Data tahunan (x3, x5-x10) — forward-carry sederhana ---
# TIDAK ada PCHIP smoothing. Data tahunan tetap step function.
# Ini jujur: PDRB tidak berubah tiap bulan, jadi kita tidak berpura-pura ada variasi.

annual_cols = [
    'x3_pdrb_per_kapita',
    'x5_penetrasi_internet_pct',
    'x6_tabungan_miliar',
    'x7_jumlah_kc_bank',
    'x8_ldr_pct',
    'x9_npl_ratio',
    'x10_rasio_umkm',
]

for col in annual_cols:
    df_filtered[col] = (
        df_filtered.groupby('provinsi_id')[col]
        .transform(lambda s: s.ffill().bfill())
    )

# --- Ringkasan missing values SETELAH cleaning ---
print('\n=== Missing Values (After Cleaning) ===')
missing_after = df_filtered.isnull().sum()
missing_after_pct = (df_filtered.isnull().sum() / len(df_filtered) * 100).round(1)
summary_after = pd.DataFrame({'missing': missing_after, 'pct': missing_after_pct})
display(summary_after[summary_after['missing'] > 0])

# Urut dan simpan
df_filtered = df_filtered.sort_values(by=['provinsi_id', 'tanggal']).reset_index(drop=True)
assert (df_filtered['provinsi_id'] == 19).sum() == 0, 'provinsi_id 19 still present'

df_filtered.to_csv(output_path, index=False)
print(f'\nSaved: {output_path}')
print(f'Final shape: {df_filtered.shape[0]:,} rows x {df_filtered.shape[1]} cols')
print(f'Unique provinces: {df_filtered["provinsi_id"].nunique()}')